<a href="https://colab.research.google.com/github/yhshengjy/ClinPKPD/blob/main/Notebook5_%E4%B8%87%E5%8F%A4%E9%9C%89%E7%B4%A0_PKPD%E6%A8%A1%E6%8B%9F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 5：万古霉素 PK/PD 模拟

本 Notebook 是临床药学 PK/PD 交互式模拟平台中“抗菌药物 PK/PD 应用”部分的第一个模块。

本节以 **万古霉素（vancomycin）** 为例，学习如何将药代动力学参数、给药方案、病原菌 MIC 和 PK/PD 指标连接起来。

万古霉素常用于严重革兰阳性菌感染，尤其是 MRSA 相关感染。对严重 MRSA 感染，临床上常用的 PK/PD 指标是：

$$
AUC_{24}/MIC
$$

2020 年 ASHP/IDSA/PIDS/SIDP 共识指南建议，对于疑似或确诊严重 MRSA 感染，万古霉素日 AUC 通常应维持在 **400--600 mg·h/L** 范围内，以兼顾疗效和肾毒性风险。该建议通常假设 MIC 为 1 mg/L。

本 Notebook 的核心逻辑是：

$$
Dose\ regimen \rightarrow Concentration(t) \rightarrow AUC_{24} \rightarrow AUC_{24}/MIC \rightarrow Target\ attainment
$$

注意：本 Notebook 用于教学模拟，不用于真实患者处方决策。真实临床给药需要结合感染部位、病原学、MIC 测定方法、肾功能、体重、血药浓度监测、合并用药和所在医院规范。

## 1. 学习目标

完成本 Notebook 后，你应该能够：


1. 解释万古霉素 PK/PD 指标 AUC24/MIC 及其与抗菌疗效的关系。
2. 理解为什么单纯谷浓度不能完全代表万古霉素暴露量。
3. 使用一室间歇静脉输注模型模拟万古霉素浓度-时间曲线。
4. 计算并解释不同给药方案下的 $C_{max}$、$C_{min}$、AUC24 和 AUC24/MIC。
5. 分析剂量、给药间隔、清除率、分布容积、肾功能和 MIC 如何共同影响疗效和安全性。

## 2. 万古霉素 PK/PD 的核心概念

万古霉素的临床 PK/PD 评价重点通常不是单次峰浓度，而是 24 小时药物暴露量与 MIC 的比值：

$$
AUC_{24}/MIC
$$

其中：

| 符号 | 含义 | 常用单位 |
|---|---|---|
| AUC24 | 24 小时药时曲线下面积 | mg·h/L |
| MIC | 最低抑菌浓度 | mg/L |
| AUC24/MIC | 暴露与病原菌敏感性的比值 | 无量纲，常写作 mg·h/L 除以 mg/L |

可以这样理解：

- AUC24 代表患者 24 小时内的总药物暴露。
- MIC 代表病原菌对药物的敏感性。
- MIC 越高，同样 AUC24 下 AUC24/MIC 越低。
- 当 MIC 较高时，单纯增加剂量可能会提高 AUC24/MIC，但也可能使 AUC24 超过安全范围。

在教学模拟中，我们采用以下简化判断：

| 判断项目 | 教学阈值 |
|---|---|
| 疗效相关目标 | AUC24/MIC ≥ 400 |
| 暴露安全上限 | AUC24 ≤ 600 mg·h/L |
| 理想教学区间 | MIC = 1 mg/L 时，AUC24 约 400--600 mg·h/L |

注意：这些阈值来自严重 MRSA 感染的万古霉素治疗药物监测共识，不应机械套用于所有感染类型、所有人群或所有临床场景。

## 3. 一室静脉间歇输注模型

万古霉素通常采用静脉间歇输注，而不是静脉快速推注。

在一室模型中，设：

$$
k = \frac{CL}{V_d}
$$

输注速率为：

$$
R_0 = \frac{Dose}{T_{inf}}
$$

其中：

| 符号 | 含义 | 常用单位 |
|---|---|---|
| Dose | 每次给药剂量 | mg |
| Tinf | 输注时间 | h |
| tau | 给药间隔 | h |
| Vd | 表观分布容积 | L |
| CL | 清除率 | L/h |
| k | 消除速率常数 | 1/h |
| R0 | 输注速率 | mg/h |

单次输注过程中，若给药后时间为 $t$，则：

### 输注期间：$0 \leq t \leq T_{inf}$

$$
C(t) = \frac{R_0}{CL}\left(1-e^{-kt}\right)
$$

### 输注结束后：$t > T_{inf}$

$$
C(t) = \frac{R_0}{CL}\left(1-e^{-kT_{inf}}\right)e^{-k(t-T_{inf})}
$$

多剂量给药时，可以把每一次给药产生的浓度曲线叠加起来，这叫做 **superposition principle（叠加原理）**。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, fixed

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True


def infusion_contribution(t_after_dose, dose_mg, infusion_h, vd_l, cl_l_h):
    """
    Concentration contribution from one IV infusion dose in a one-compartment model.
    """
    k_elim = cl_l_h / vd_l
    infusion_rate = dose_mg / infusion_h
    concentration = np.zeros_like(t_after_dose, dtype=float)

    during = (t_after_dose >= 0) & (t_after_dose <= infusion_h)
    after = t_after_dose > infusion_h

    concentration[during] = (
        infusion_rate / cl_l_h
        * (1 - np.exp(-k_elim * t_after_dose[during]))
    )

    concentration[after] = (
        infusion_rate / cl_l_h
        * (1 - np.exp(-k_elim * infusion_h))
        * np.exp(-k_elim * (t_after_dose[after] - infusion_h))
    )

    return concentration


def simulate_multiple_iv_infusions(
    dose_mg,
    tau_h,
    infusion_h,
    vd_l,
    cl_l_h,
    n_doses,
    dt=0.01
):
    """
    Simulate multiple intermittent IV infusions using superposition.
    """
    total_time_h = tau_h * n_doses
    t = np.arange(0, total_time_h + dt, dt)
    concentration = np.zeros_like(t, dtype=float)

    dose_times = np.arange(0, n_doses * tau_h, tau_h)

    for dose_time in dose_times:
        concentration += infusion_contribution(
            t_after_dose=t - dose_time,
            dose_mg=dose_mg,
            infusion_h=infusion_h,
            vd_l=vd_l,
            cl_l_h=cl_l_h
        )

    return t, concentration, dose_times


def calculate_vancomycin_metrics(
    t,
    concentration,
    dose_mg,
    tau_h,
    infusion_h,
    vd_l,
    cl_l_h,
    mic_mg_l,
    n_doses
):
    """
    Calculate PK/PD metrics for vancomycin teaching simulation.
    """
    k_elim = cl_l_h / vd_l
    half_life = np.log(2) / k_elim

    # Steady-state AUC24 for linear PK
    daily_dose_mg = dose_mg * (24 / tau_h)
    auc24_ss = daily_dose_mg / cl_l_h
    auc_mic = auc24_ss / mic_mg_l

    # Last dosing interval metrics
    last_dose_time = (n_doses - 1) * tau_h
    interval_mask = (t >= last_dose_time) & (t <= last_dose_time + tau_h)
    t_interval = t[interval_mask] - last_dose_time
    c_interval = concentration[interval_mask]

    cmax_last = np.max(c_interval)
    tmax_last = t_interval[np.argmax(c_interval)]
    cmin_last = c_interval[-1]
    c_end_infusion = np.interp(infusion_h, t_interval, c_interval)

    if auc_mic < 400:
        target_status = "Below efficacy target"
    elif auc24_ss > 600:
        target_status = "High exposure risk"
    else:
        target_status = "Within teaching target range"

    metrics = {
        "Elimination rate constant": k_elim,
        "Half-life": half_life,
        "Daily dose": daily_dose_mg,
        "AUC24 at steady state": auc24_ss,
        "AUC24/MIC": auc_mic,
        "Cmax in last interval": cmax_last,
        "Tmax in last interval": tmax_last,
        "Concentration at end of infusion": c_end_infusion,
        "Cmin before next dose": cmin_last,
        "Target status": target_status
    }

    return metrics


def make_metrics_table(metrics):
    """
    Format metrics dictionary as a dataframe.
    """
    rows = []
    for key, value in metrics.items():
        if isinstance(value, str):
            formatted = value
        elif "rate constant" in key:
            formatted = f"{value:.4f} 1/h"
        elif "Half-life" in key:
            formatted = f"{value:.2f} h"
        elif "Daily dose" in key:
            formatted = f"{value:.0f} mg/day"
        elif "AUC" in key and "MIC" not in key:
            formatted = f"{value:.1f} mg*h/L"
        elif "AUC24/MIC" in key:
            formatted = f"{value:.1f}"
        else:
            formatted = f"{value:.2f} mg/L"
        rows.append({"Metric": key, "Value": formatted})
    return pd.DataFrame(rows)

## 4. 交互模拟 1：万古霉素多剂量浓度曲线与 AUC/MIC

下面模拟万古霉素静脉间歇输注后的多剂量浓度--时间曲线。

可以调节：

- 每次剂量 Dose
- 给药间隔 tau
- 输注时间 Tinf
- 分布容积 Vd
- 清除率 CL
- MIC
- 给药次数

请重点观察：

- 给药多次后，浓度是否逐渐接近稳态？
- 剂量增加时，AUC24 和 AUC24/MIC 如何变化？
- CL 降低时，AUC24 和谷浓度如何变化？
- MIC 增加时，即使 AUC24 不变，AUC24/MIC 是否下降？

In [ ]:
def plot_vancomycin_pkpd(
    dose_mg=1000,
    tau_h=12,
    infusion_h=1,
    vd_l=70,
    cl_l_h=4,
    mic_mg_l=1,
    n_doses=8
):
    if infusion_h >= tau_h:
        print("Infusion time must be shorter than dosing interval.")
        return

    t, concentration, dose_times = simulate_multiple_iv_infusions(
        dose_mg=dose_mg,
        tau_h=tau_h,
        infusion_h=infusion_h,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        n_doses=n_doses,
        dt=0.01
    )

    metrics = calculate_vancomycin_metrics(
        t=t,
        concentration=concentration,
        dose_mg=dose_mg,
        tau_h=tau_h,
        infusion_h=infusion_h,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        mic_mg_l=mic_mg_l,
        n_doses=n_doses
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(t, concentration, linewidth=2, label="Vancomycin concentration")
    ax.axhline(mic_mg_l, linestyle="--", label=f"MIC = {mic_mg_l:.1f} mg/L")

    for dose_time in dose_times:
        ax.axvline(dose_time, alpha=0.15)

    ax.set_title("Vancomycin Multiple IV Infusion Simulation")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    display(make_metrics_table(metrics))


interact(
    plot_vancomycin_pkpd,
    dose_mg=FloatSlider(value=1000, min=250, max=2500, step=250, description="Dose"),
    tau_h=Dropdown(options=[6, 8, 12, 24, 36, 48], value=12, description="Tau"),
    infusion_h=FloatSlider(value=1, min=0.5, max=4, step=0.5, description="Tinf"),
    vd_l=FloatSlider(value=70, min=30, max=150, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=4, min=0.5, max=10, step=0.5, description="CL"),
    mic_mg_l=Dropdown(options=[0.5, 1, 1.5, 2], value=1, description="MIC"),
    n_doses=IntSlider(value=8, min=2, max=20, step=1, description="Doses")
);

interactive(children=(FloatSlider(value=1000.0, description='Dose', max=2500.0, min=250.0, step=250.0), Dropdo…

## 5. 观察任务 1：剂量、间隔、CL 与 MIC 的影响

请完成以下操作。

### 任务 A：标准方案

设置：

- Dose = 1000 mg
- Tau = 12 h
- Tinf = 1 h
- Vd = 70 L
- CL = 4 L/h
- MIC = 1 mg/L
- Doses = 8

记录：

- AUC24 at steady state
- AUC24/MIC
- Cmax in last interval
- Cmin before next dose
- Target status

### 任务 B：剂量增加

只将 Dose 改为 1500 mg。

观察：

- AUC24 是否升高？
- AUC24/MIC 是否升高？
- Cmin 是否升高？
- Target status 是否提示高暴露风险？

### 任务 C：清除率下降

将 Dose 恢复为 1000 mg，只将 CL 改为 2 L/h。

观察：

- AUC24 是否明显升高？
- 半衰期是否延长？
- Cmin 是否升高？
- 这是否可以模拟肾功能下降患者的风险？

### 任务 D：MIC 升高

将 CL 恢复为 4 L/h，只将 MIC 改为 2 mg/L。

观察：

- AUC24 是否改变？
- AUC24/MIC 是否下降？
- 如果想让 AUC24/MIC ≥ 400，需要 AUC24 达到多少？
- 这个 AUC24 是否超过 600 mg·h/L？

## 6. 为什么 MIC 很重要？

对于万古霉素，AUC24/MIC 同时受到两个因素影响：

$$
AUC_{24}/MIC = \frac{AUC_{24}}{MIC}
$$

如果 AUC24 保持不变：

- MIC = 0.5 mg/L 时，AUC24/MIC 较高。
- MIC = 1 mg/L 时，AUC24/MIC 为 AUC24 的数值。
- MIC = 2 mg/L 时，AUC24/MIC 只有 MIC = 1 mg/L 时的一半。

这解释了为什么病原菌 MIC 升高时，同样给药方案可能不再达标。

例如，如果目标是：

$$
AUC_{24}/MIC \geq 400
$$

则需要：

$$
AUC_{24} \geq 400 \times MIC
$$

当 MIC = 2 mg/L 时，需要：

$$
AUC_{24} \geq 800\ mg \cdot h/L
$$

这已经超过常用的 600 mg·h/L 暴露上限，因此从教学角度看，MIC = 2 mg/L 时可能很难通过简单增加万古霉素剂量来同时满足疗效和安全性。

In [ ]:
def plot_mic_effect(
    auc24=500
):
    mic_values = np.array([0.5, 1.0, 1.5, 2.0])
    auc_mic_values = auc24 / mic_values
    required_auc_values = 400 * mic_values

    result = pd.DataFrame({
        "MIC (mg/L)": mic_values,
        "AUC24 (mg*h/L)": [auc24] * len(mic_values),
        "AUC24/MIC": auc_mic_values,
        "AUC24 required for AUC/MIC >= 400": required_auc_values,
        "Feasible within AUC <= 600?": ["Yes" if x <= 600 else "No" for x in required_auc_values]
    })

    fig, ax = plt.subplots()
    ax.bar([str(x) for x in mic_values], auc_mic_values)
    ax.axhline(400, linestyle="--", label="AUC/MIC target = 400")
    ax.set_title("Effect of MIC on AUC24/MIC")
    ax.set_xlabel("MIC (mg/L)")
    ax.set_ylabel("AUC24/MIC")
    ax.legend()
    plt.show()

    display(result)


interact(
    plot_mic_effect,
    auc24=FloatSlider(value=500, min=200, max=900, step=50, description="AUC24")
);

interactive(children=(FloatSlider(value=500.0, description='AUC24', max=900.0, min=200.0, step=50.0), Output()…

## 7. 观察任务 2：同样 AUC24，不同 MIC

请使用上面的 MIC 模拟完成以下任务。

### 任务 A

设置：

- AUC24 = 500 mg·h/L

记录不同 MIC 下的 AUC24/MIC：

- MIC = 0.5 mg/L
- MIC = 1 mg/L
- MIC = 1.5 mg/L
- MIC = 2 mg/L

### 任务 B

将 AUC24 调为 600 mg·h/L。

思考：

- MIC = 1 mg/L 时是否可以达到 AUC24/MIC ≥ 400？
- MIC = 2 mg/L 时是否可以达到 AUC24/MIC ≥ 400？
- 如果不能，是否应该无限制增加万古霉素剂量？为什么？

## 8. 肾功能变化与万古霉素暴露

万古霉素主要经肾脏清除，因此肾功能变化会显著影响 CL、半衰期、AUC24 和谷浓度。

本节使用 Cockcroft--Gault 公式估算肌酐清除率：

### 男性

$$
CrCl = \frac{(140-age) \times weight}{72 \times SCr}
$$

### 女性

$$
CrCl = 0.85 \times \frac{(140-age) \times weight}{72 \times SCr}
$$

其中：

| 符号 | 含义 | 单位 |
|---|---|---|
| age | 年龄 | years |
| weight | 体重 | kg |
| SCr | 血清肌酐 | mg/dL |
| CrCl | 肌酐清除率 | mL/min |

为了教学演示，我们使用一个简化关系估算万古霉素清除率：

$$
CL_{vanco} = 0.5 + 0.06 \times CrCl
$$

该公式仅用于本 Notebook 的教学模拟，不是临床验证的万古霉素剂量计算公式。

同时假设：

$$
V_d = 0.7 \times weight
$$

真实临床中，万古霉素剂量应根据血药浓度监测和患者个体情况动态调整。

In [ ]:
def cockcroft_gault(age_years, weight_kg, scr_mg_dl, sex):
    """
    Estimate creatinine clearance using Cockcroft-Gault equation.
    """
    crcl = ((140 - age_years) * weight_kg) / (72 * scr_mg_dl)
    if sex == "Female":
        crcl *= 0.85
    return crcl


def estimate_vanco_cl_from_crcl(crcl_ml_min):
    """
    Teaching-only vancomycin clearance model.
    Not intended for clinical dosing.
    """
    cl_l_h = 0.5 + 0.06 * crcl_ml_min
    return max(cl_l_h, 0.3)


def plot_renal_function_vanco(
    sex="Male",
    age_years=65,
    weight_kg=70,
    scr_mg_dl=1.0,
    dose_mg=1000,
    tau_h=12,
    infusion_h=1,
    mic_mg_l=1,
    n_doses=8
):
    if infusion_h >= tau_h:
        print("Infusion time must be shorter than dosing interval.")
        return

    crcl = cockcroft_gault(
        age_years=age_years,
        weight_kg=weight_kg,
        scr_mg_dl=scr_mg_dl,
        sex=sex
    )

    vd_l = 0.7 * weight_kg
    cl_l_h = estimate_vanco_cl_from_crcl(crcl)

    t, concentration, dose_times = simulate_multiple_iv_infusions(
        dose_mg=dose_mg,
        tau_h=tau_h,
        infusion_h=infusion_h,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        n_doses=n_doses,
        dt=0.01
    )

    metrics = calculate_vancomycin_metrics(
        t=t,
        concentration=concentration,
        dose_mg=dose_mg,
        tau_h=tau_h,
        infusion_h=infusion_h,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        mic_mg_l=mic_mg_l,
        n_doses=n_doses
    )

    target_auc = 500
    suggested_daily_dose = target_auc * cl_l_h
    suggested_dose_per_interval = suggested_daily_dose / (24 / tau_h)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(t, concentration, linewidth=2, label="Predicted concentration")
    ax.axhline(mic_mg_l, linestyle="--", label=f"MIC = {mic_mg_l:.1f} mg/L")

    for dose_time in dose_times:
        ax.axvline(dose_time, alpha=0.15)

    ax.set_title("Vancomycin Exposure and Renal Function")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    patient_table = pd.DataFrame({
        "Parameter": [
            "Sex",
            "Age",
            "Weight",
            "SCr",
            "Estimated CrCl",
            "Teaching Vd estimate",
            "Teaching CL estimate",
            "Current dose",
            "Current interval",
            "Suggested dose per interval for AUC24 ≈ 500"
        ],
        "Value": [
            sex,
            f"{age_years:.0f} years",
            f"{weight_kg:.1f} kg",
            f"{scr_mg_dl:.2f} mg/dL",
            f"{crcl:.1f} mL/min",
            f"{vd_l:.1f} L",
            f"{cl_l_h:.2f} L/h",
            f"{dose_mg:.0f} mg",
            f"q{tau_h:.0f}h",
            f"{suggested_dose_per_interval:.0f} mg q{tau_h:.0f}h"
        ]
    })

    display(patient_table)
    display(make_metrics_table(metrics))


interact(
    plot_renal_function_vanco,
    sex=Dropdown(options=["Male", "Female"], value="Male", description="Sex"),
    age_years=FloatSlider(value=65, min=18, max=95, step=1, description="Age"),
    weight_kg=FloatSlider(value=70, min=40, max=140, step=5, description="Weight"),
    scr_mg_dl=FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1, description="SCr"),
    dose_mg=FloatSlider(value=1000, min=250, max=2500, step=250, description="Dose"),
    tau_h=Dropdown(options=[6, 8, 12, 24, 36, 48], value=12, description="Tau"),
    infusion_h=FloatSlider(value=1, min=0.5, max=4, step=0.5, description="Tinf"),
    mic_mg_l=Dropdown(options=[0.5, 1, 1.5, 2], value=1, description="MIC"),
    n_doses=IntSlider(value=8, min=2, max=20, step=1, description="Doses")
);

interactive(children=(Dropdown(description='Sex', options=('Male', 'Female'), value='Male'), FloatSlider(value…

## 9. 观察任务 3：肾功能下降对暴露的影响

请使用上面的肾功能模拟完成以下任务。

### 任务 A：肾功能相对正常患者

设置：

- Sex = Male
- Age = 40 years
- Weight = 70 kg
- SCr = 0.9 mg/dL
- Dose = 1000 mg
- Tau = 12 h
- MIC = 1 mg/L

记录：

- Estimated CrCl
- Teaching CL estimate
- AUC24
- Cmin before next dose
- Target status

### 任务 B：肾功能下降患者

只将参数改为：

- Age = 80 years
- SCr = 2.0 mg/dL

观察：

- Estimated CrCl 是否下降？
- Teaching CL estimate 是否下降？
- AUC24 是否升高？
- Cmin 是否升高？
- 是否提示需要调整剂量或延长间隔？

### 任务 C：剂量调整思考

观察表格中：

- Suggested dose per interval for AUC24 ≈ 500

思考：

- 为什么肾功能下降时建议剂量会降低？
- 为什么这个建议不能直接替代真实 TDM？
- 真实临床还需要哪些信息？

## 10. 给药方案比较：同样日剂量，不同给药间隔

对于线性 PK，如果总日剂量相同，并且 CL 不变，则 AUC24 通常相近。

但是，给药间隔不同会改变浓度波动：

- 给药间隔短：峰谷波动较小。
- 给药间隔长：峰谷波动较大。
- 总日剂量相同：AUC24 可能相近，但 Cmax 和 Cmin 可以不同。

这有助于理解：

> AUC、Cmax 和 Cmin 代表不同的临床信息。

AUC 更反映总暴露，Cmin 更容易受给药间隔和清除率影响。

In [ ]:
def compare_regimens_same_daily_dose(
    daily_dose_mg=2000,
    infusion_h=1,
    vd_l=70,
    cl_l_h=4,
    mic_mg_l=1,
    n_days=4
):
    regimens = [
        {"name": "q8h", "tau_h": 8, "dose_mg": daily_dose_mg / 3},
        {"name": "q12h", "tau_h": 12, "dose_mg": daily_dose_mg / 2},
        {"name": "q24h", "tau_h": 24, "dose_mg": daily_dose_mg}
    ]

    fig, ax = plt.subplots(figsize=(10, 5))
    rows = []

    for regimen in regimens:
        tau_h = regimen["tau_h"]
        dose_mg = regimen["dose_mg"]
        n_doses = int((24 * n_days) / tau_h)

        t, concentration, dose_times = simulate_multiple_iv_infusions(
            dose_mg=dose_mg,
            tau_h=tau_h,
            infusion_h=infusion_h,
            vd_l=vd_l,
            cl_l_h=cl_l_h,
            n_doses=n_doses,
            dt=0.01
        )

        metrics = calculate_vancomycin_metrics(
            t=t,
            concentration=concentration,
            dose_mg=dose_mg,
            tau_h=tau_h,
            infusion_h=infusion_h,
            vd_l=vd_l,
            cl_l_h=cl_l_h,
            mic_mg_l=mic_mg_l,
            n_doses=n_doses
        )

        ax.plot(t, concentration, linewidth=2, label=f"{dose_mg:.0f} mg {regimen['name']}")

        rows.append({
            "Regimen": f"{dose_mg:.0f} mg {regimen['name']}",
            "Daily dose": f"{daily_dose_mg:.0f} mg/day",
            "AUC24": f"{metrics['AUC24 at steady state']:.1f} mg*h/L",
            "AUC24/MIC": f"{metrics['AUC24/MIC']:.1f}",
            "Cmax last interval": f"{metrics['Cmax in last interval']:.2f} mg/L",
            "Cmin before next dose": f"{metrics['Cmin before next dose']:.2f} mg/L",
            "Target status": metrics["Target status"]
        })

    ax.axhline(mic_mg_l, linestyle="--", label=f"MIC = {mic_mg_l:.1f} mg/L")
    ax.set_title("Comparison of Regimens with the Same Daily Dose")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    display(pd.DataFrame(rows))


interact(
    compare_regimens_same_daily_dose,
    daily_dose_mg=FloatSlider(value=2000, min=1000, max=4000, step=250, description="Daily dose"),
    infusion_h=FloatSlider(value=1, min=0.5, max=4, step=0.5, description="Tinf"),
    vd_l=FloatSlider(value=70, min=30, max=150, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=4, min=0.5, max=10, step=0.5, description="CL"),
    mic_mg_l=Dropdown(options=[0.5, 1, 1.5, 2], value=1, description="MIC"),
    n_days=IntSlider(value=4, min=2, max=7, step=1, description="Days")
);

interactive(children=(FloatSlider(value=2000.0, description='Daily dose', max=4000.0, min=1000.0, step=250.0),…

## 11. 观察任务 4：同样日剂量，不同给药间隔

请使用上面的方案比较模拟完成以下任务。

### 任务 A

设置：

- Daily dose = 2000 mg/day
- Tinf = 1 h
- Vd = 70 L
- CL = 4 L/h
- MIC = 1 mg/L

比较：

- q8h
- q12h
- q24h

观察：

- 三个方案的 AUC24 是否相近？
- Cmax 是否相同？
- Cmin 是否相同？
- 哪个方案峰谷波动最大？

### 任务 B

将 CL 降低到 2 L/h。

观察：

- AUC24 是否升高？
- Cmin 是否明显升高？
- 在清除率下降时，为什么给药间隔可能需要延长？

## 12. 从 AUC24 到 AUC/MIC：综合判断

万古霉素 PK/PD 判断不能只看一个数字。

教学中可以采用以下顺序：

1. 先看 AUC24 是否达到疗效相关暴露。
2. 再看 AUC24/MIC 是否达到目标。
3. 再看 AUC24 是否超过安全上限。
4. 同时观察 Cmin 是否明显升高。
5. 结合肾功能、感染严重程度和 TDM 结果判断是否需要调整。

可以用下面的简化流程理解：

$$
AUC_{24}/MIC < 400 \Rightarrow 可能暴露不足
$$

$$
400 \leq AUC_{24} \leq 600\quad (MIC=1) \Rightarrow 教学目标范围
$$

$$
AUC_{24} > 600 \Rightarrow 暴露偏高，肾毒性风险可能增加
$$

但请注意：

> 真实临床中，万古霉素 AUC 通常需要通过血药浓度监测和药动学方法估算，而不是只靠本 Notebook 的简化模型。

## 13. 自测题：万古霉素 PK/PD

请根据本 Notebook 的内容完成以下自测题。建议先独立作答，再查看下一单元格中的参考答案。

---

### 题目 1：万古霉素严重 MRSA 感染常用的 PK/PD 指标是什么？

A. Cmax/MIC  
B. AUC24/MIC  
C. %fT > MIC  
D. Tmax/MIC  

---

### 题目 2：当 MIC 从 1 mg/L 升高到 2 mg/L，而 AUC24 不变时，AUC24/MIC 会怎样变化？

A. 增加 2 倍  
B. 不变  
C. 降低为原来的一半  
D. 降低为原来的四分之一  

---

### 题目 3：对于线性 PK，如果总日剂量和 CL 不变，q8h、q12h 和 q24h 给药方案的 AUC24 通常如何？

A. q8h 的 AUC24 一定最大  
B. q24h 的 AUC24 一定最大  
C. AUC24 通常相近，但峰谷波动不同  
D. 三者 Cmax 和 Cmin 都完全相同  

---

### 题目 4：肾功能下降导致万古霉素 CL 降低时，以下哪项最可能发生？

A. AUC24 降低，半衰期缩短  
B. AUC24 升高，半衰期延长  
C. MIC 自动降低  
D. 药物吸收速率 ka 增大  

---

### 题目 5：如果 MIC = 2 mg/L，为达到 AUC24/MIC ≥ 400，理论上需要的 AUC24 至少是多少？

A. 200 mg·h/L  
B. 400 mg·h/L  
C. 600 mg·h/L  
D. 800 mg·h/L

## 14. 自测题参考答案

### 题目 1

**参考答案：B**

**解析：**  
万古霉素治疗严重 MRSA 感染时，常用 PK/PD 指标为 AUC24/MIC。AUC24 反映 24 小时总暴露，MIC 反映病原菌敏感性。

---

### 题目 2

**参考答案：C**

**解析：**  

$$
AUC_{24}/MIC = \frac{AUC_{24}}{MIC}
$$

当 AUC24 不变，而 MIC 从 1 增加到 2 时，分母增加 2 倍，因此 AUC24/MIC 降低为原来的一半。

---

### 题目 3

**参考答案：C**

**解析：**  
在线性 PK 中，稳态 AUC24 主要由总日剂量和 CL 决定：

$$
AUC_{24} = \frac{Daily\ Dose}{CL}
$$

如果总日剂量和 CL 相同，AUC24 通常相近。但不同给药间隔会改变 Cmax、Cmin 和峰谷波动。

---

### 题目 4

**参考答案：B**

**解析：**  
肾功能下降时，万古霉素清除率可能降低。CL 降低会使消除变慢、半衰期延长，并导致 AUC24 和谷浓度升高。

---

### 题目 5

**参考答案：D**

**解析：**  
如果目标为：

$$
AUC_{24}/MIC \geq 400
$$

当 MIC = 2 mg/L 时：

$$
AUC_{24} \geq 400 \times 2 = 800\ mg \cdot h/L
$$

这超过常用的 600 mg·h/L 暴露上限，因此 MIC = 2 mg/L 时可能难以通过单纯增加万古霉素剂量同时兼顾疗效和安全性。

## 15. 本 Notebook 小结

本 Notebook 通过万古霉素示例介绍了抗菌药物 PK/PD 指标 AUC24/MIC 的教学模拟。

你应该掌握以下核心结论：


1. AUC24/MIC 是万古霉素的重要 PK/PD 指标，尤其常用于严重 MRSA 感染相关的暴露评价。
2. AUC24 反映 24 小时内的总药物暴露量，而 MIC 反映病原体对药物的敏感性。
3. 谷浓度与药物暴露有关，但不能完全替代基于 AUC 的评价。
4. 剂量、给药间隔、输注时间、清除率、分布容积、肾功能和 MIC 共同决定万古霉素暴露水平。
5. MIC 升高会使同一给药方案更难在不超过安全暴露范围的情况下达到 PK/PD 目标。
6. 简化 PK/PD 模拟有助于理解给药逻辑，但不能替代临床 TDM 和个体化剂量调整。

本节的完整逻辑可以概括为：

$$
Dose\ regimen \rightarrow Concentration(t) \rightarrow AUC_{24} \rightarrow AUC_{24}/MIC \rightarrow Efficacy/Safety
$$

下一节将进一步学习：

> β-内酰胺类抗菌药物的 %fT > MIC 模拟。

## 16. 参考资料

1. Rybak MJ, Le J, Lodise TP, et al. Therapeutic monitoring of vancomycin for serious methicillin-resistant *Staphylococcus aureus* infections: A revised consensus guideline and review by ASHP, IDSA, PIDS, and SIDP. 2020.

2. DailyMed. Vancomycin Hydrochloride for Injection, USP. U.S. National Library of Medicine.

3. Cockcroft DW, Gault MH. Prediction of creatinine clearance from serum creatinine. *Nephron*. 1976.

说明：本 Notebook 的公式和代码用于教学演示，涉及万古霉素清除率估算的部分为简化模型，不用于真实处方或 TDM 决策。